# Reranking

Reranking is the process of **reordering the documents retrieved by the initial retriever based on their relevance to the query**.

### Flow

Query → Retriever → Top-K Documents → Reranker → Most Relevant Documents → LLM

### Why use Reranking?

- Improves retrieval accuracy
- Removes less relevant documents
- Provides better context to the LLM
- Reduces irrelevant information

### Example

If the retriever returns 10 documents, the reranker scores them again and selects the **most relevant 3–5 documents** for the LLM.

**In short:**

> Reranking improves RAG retrieval by reordering initially retrieved documents according to their relevance to the user's query.
> aasma chain query direct db bata lenxa


In [15]:
#### INDEXING ####

# Blog load gareko
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

blog_docs = loader.load()


# Document lai sano sano chunks ma divide gareko
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, chunk_overlap=50
)

splits = text_splitter.split_documents(blog_docs)

print(f"Total chunks: {len(splits)}")


# Free/local embedding model use gareko
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")


# Chunks lai Chroma vector database ma store gareko
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)


# Vector database lai retriever ma convert gareko
retriever = vectorstore.as_retriever()

print("Indexing complete!")

Total chunks: 50
Indexing complete!


In [16]:
from langchain_core.prompts import ChatPromptTemplate

# RAG-Fusion ko lagi prompt banayeko
# Euta question bata multiple related search queries generate garna use huncha

template = """You are a helpful assistant that generates multiple search queries based on a single input query.

Generate multiple search queries related to: {question}

Output (4 queries):"""

# Template lai ChatPromptTemplate ma convert gareko
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# Local Ollama model use gareko
llm = ChatOllama(model="llama3:latest", temperature=0)

# RAG-Fusion ko lagi multiple search queries generate garne chain
generate_queries = (
    prompt_rag_fusion | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

In [18]:
from langchain_core.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    """
    Multiple retrieval results lai combine garera
    documents lai naya relevance score ko basis ma rerank garne.

    k = RRF formula ko constant ho.
    """

    # Pratyek unique document ko fused score store garne dictionary
    fused_scores = {}

    # Multiple query bata aayeko retrieval results ma loop garne
    for docs in results:

        # Document ko rank/position anusar loop garne
        for rank, doc in enumerate(docs):

            # Document lai string ma convert garera unique key banaune
            doc_str = dumps(doc)

            # Document first time aayeko ho bhane score 0 bata start garne
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0

            # RRF formula bata document ko score update garne
            # Higher-ranked document lai higher score milcha
            fused_scores[doc_str] += 1 / (rank + k)

    # Fused score ko descending order ma documents sort garne
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Reranked documents ra uniharuko scores return garne
    return reranked_results

In [19]:
from langchain_core.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    """
    Multiple retrieval lists lai combine garera
    documents lai relevance ko basis ma rerank garne.

    k = RRF formula ko constant ho.
    """

    # Pratyek unique document ko fused score store garne
    fused_scores = {}

    # Sabai retrieval result lists ma loop garne
    for docs in results:

        # Euta list bhitra ko documents ma loop garne
        # rank = document ko position
        for rank, doc in enumerate(docs):

            # Document lai string ma convert garera unique key banaune
            doc_str = dumps(doc)

            # Document pahilo choti aayeko cha bhane score 0 bata start garne
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0

            # RRF formula bata score calculate garne
            fused_scores[doc_str] += 1 / (rank + k)

    # Fused score ko descending order ma documents sort garne
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Reranked documents ra score return garne
    return reranked_results


# User ko question
question = "What is task decomposition for LLM agents?"


# RAG-Fusion pipeline:
# 1. Multiple queries generate garne
# 2. Pratyek query bata documents retrieve garne
# 3. RRF use garera sabai results combine + rerank garne
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion


# RAG-Fusion chain run garne
docs = retrieval_chain_rag_fusion.invoke({"question": question})


# Final reranked documents ko number
print(f"Total reranked documents: {len(docs)}")

Total reranked documents: 4


C:\Users\Acer\AppData\Local\Temp\ipykernel_26960\1472209112.py:34: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  (loads(doc), score)
C:\Users\Acer\AppData\Local\Temp\ipykernel_26960\1472209112.py:34: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


In [20]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# RAG ko lagi prompt banayeko
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)


# Free/local Ollama model use gareko
llm = ChatOllama(model="llama3:latest", temperature=0)


# Final RAG-Fusion chain
# 1. RAG-Fusion bata relevant documents retrieve garne
# 2. Context ra question prompt ma pass garne
# 3. Llama 3 le final answer generate garne
# 4. Output lai string ma convert garne

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)


# Final RAG-Fusion chain run gareko
final_answer = final_rag_chain.invoke({"question": question})

print(final_answer)

C:\Users\Acer\AppData\Local\Temp\ipykernel_26960\1472209112.py:34: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


According to the provided context, task decomposition for LLM (Large Language Model) agents involves breaking down large tasks into smaller, manageable subgoals. This enables efficient handling of complex tasks and can be achieved through various methods such as:

1. Simple prompting: Using prompts like "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?"
2. Task-specific instructions: Providing task-specific instructions, such as "Write a story outline" for writing a novel.
3. Human inputs: Utilizing human inputs to guide the decomposition process.

Task decomposition can be further enhanced through techniques like Chain of Thought (CoT) and Tree of Thoughts (Yao et al., 2023), which involve decomposing problems into multiple thought steps, generating multiple thoughts per step, and creating a tree structure.


In [21]:
from sentence_transformers import CrossEncoder

# Local reranker model load gareko
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5749.52it/s]


In [ ]:
from sentence_transformers import CrossEncoder

# Local CrossEncoder reranker model load gareko
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Pahila vector search bata Top 10 documents retrieve garne
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# User ko question bata documents retrieve gareko
retrieved_docs = retriever.invoke(question)

# Query ra each document ko pair banaune
pairs = [(question, doc.page_content) for doc in retrieved_docs]

# CrossEncoder le pratyek document ko relevance score calculate garne
scores = reranker.predict(pairs)

# Document ra score lai pair garera sort garne
ranked_docs = sorted(zip(retrieved_docs, scores), key=lambda x: x[1], reverse=True)

# Final reranked documents
compressed_docs = [doc for doc, score in ranked_docs]

# Top 5 documents herne
for i, (doc, score) in enumerate(ranked_docs[:5], 1):f
    print(f"\nRank {i} | Score: {score:.4f}")
    print(doc.page_content[:500])

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8049.90it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Rank 1 | Score: 3.7921
LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In a LLM-p

Rank 2 | Score: 3.7921
LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general 